In [64]:
import re
import string
from collections import Counter
import pandas as pd
import spacy
import textstat

In [19]:
# Загружаем словарь один раз при старте программы
oxford_dict = pd.read_csv('oxford-5k.csv').set_index('word')['level'].to_dict()
oxford_dict

{'a': 'a1',
 'abandon': 'b2',
 'ability': 'a2',
 'able': 'a2',
 'abolish': 'c1',
 'abortion': 'c1',
 'about': 'a1',
 'above': 'a1',
 'abroad': 'a2',
 'absence': 'c1',
 'absent': 'c1',
 'absolute': 'b2',
 'absolutely': 'b1',
 'absorb': 'b2',
 'abstract': 'b2',
 'absurd': 'c1',
 'abundance': 'c1',
 'abuse': 'c1',
 'academic': 'b2',
 'academy': 'c1',
 'accelerate': 'c1',
 'accent': 'b2',
 'accept': 'a2',
 'acceptable': 'b2',
 'acceptance': 'c1',
 'access': 'b1',
 'accessible': 'c1',
 'accident': 'a2',
 'accidentally': 'b2',
 'accommodate': 'b2',
 'accommodation': 'b1',
 'accompany': 'b2',
 'accomplish': 'b2',
 'accomplishment': 'c1',
 'accordance': 'c1',
 'according to': 'a2',
 'accordingly': 'c1',
 'account': 'b2',
 'accountability': 'c1',
 'accountable': 'c1',
 'accountant': 'b2',
 'accounting': nan,
 'accumulate': 'c1',
 'accumulation': 'c1',
 'accuracy': 'b2',
 'accurate': 'b2',
 'accurately': 'b2',
 'accusation': 'c1',
 'accuse': 'b2',
 'accused': 'c1',
 'achieve': 'a2',
 'achievemen

In [60]:
nlp = spacy.load("en_core_web_sm")

In [111]:
def analyze_text(text):
    # Предварительная обработка текста
    text = text.lower()
    # Удаляем пунктуацию
    text_clean = text.translate(str.maketrans('', '', string.punctuation))
    words = text_clean.split()
    sentences = re.split(r'[.!?]+', text)
    sentences = [s for s in sentences if s.strip()]  # Убираем пустые строки

    # 1. Общее количество слов
    total_words = len(words)

    # 2. Средняя длина слова (в буквах)
    if total_words > 0:
        avg_word_length = sum(len(word) for word in words) / total_words
    else:
        avg_word_length = 0

    # 3. Словарный запас: количество уникальных слов
    unique_words = set(words)
    vocabulary_size = len(unique_words)

    # 4. Лексическое разнообразие: отношение уникальных слов к общему числу
    if total_words > 0:
        lexical_diversity = vocabulary_size / total_words
    else:
        lexical_diversity = 0

    # Подсчёт слов по уровням
    level_counts = {'a1': 0, 'a2': 0, 'b1': 0, 'b2': 0, 'c1': 0, 'c2': 0}
    for word in words:
        if word in oxford_dict:
            level = oxford_dict[word]
            if level in level_counts:
                level_counts[level] += 1
    
    # Вычисляем доли
    if total_words > 0:
        for level in level_counts:
            level_counts[level] = round(level_counts[level] / total_words, 2)

        doc = nlp(text)

    # Считаем сложные конструкции
    complex_sentences = 0
    passive_constructions = 0

    for sent in doc.sents:
        # Признаки сложного предложения: подчинительные союзы, относительные местоимения
        has_subordinate = any(token.dep_ in ['mark', 'advcl'] for token in sent)
        has_relative = any(token.dep_ == 'relcl' for token in sent)

        if has_subordinate or has_relative:
            complex_sentences += 1

        # Пассивный залог: глагол to be + V3
        for token in sent:
            if token.tag_ == 'VBN' and token.head.lemma_ == 'be':
                passive_constructions += 1

    # Собираем результаты
    results = {
        'total_words': total_words,
        'avg_word_length': round(avg_word_length, 2),
        'A1_level_words': level_counts['a1'],
        'A2_level_words': level_counts['a2'],
        'B1_level_words': level_counts['b1'],
        'B2_level_words': level_counts['b2'],
        'C1_level_words': level_counts['c1'],
        'C2_level_words': level_counts['c2'],
        'vocabulary_size': vocabulary_size,
        'lexical_diversity': round(lexical_diversity, 2),
        'complex_sentence_ratio': complex_sentences / len(list(doc.sents)) if doc.sents else 0,
        'passive_voice_ratio': passive_constructions / total_words if total_words else 0,
        'flesch_reading_ease': textstat.flesch_reading_ease(text),
        'smog_index': textstat.smog_index(text),
        'coleman_liau_index': textstat.coleman_liau_index(text),
        'sentence_count': textstat.sentence_count(text),
        'lexicon_count': textstat.lexicon_count(text, removepunct=True)
    }
    return results

In [130]:
def estimate_level(results):
    score = 0
    #лексика
    if results['B2_level_words'] >= 0.05 or results['C1_level_words'] >= 0.03 or results['C2_level_words'] >= 0.01:
        score += 2
    elif results['B1_level_words'] >= 0.1:
        score += 1

    #грамматика
    if results.get('complex_sentence_ratio', 0) >= 0.3:
        score += 3
    if results.get('passive_voice_ratio', 0) > 0.02:
        score += 2

    #cтилистика
    if results.get('flesch_reading_ease', 100) >= 60:  # Более сложный текст
        score += 3   

    #традиционные метрики
    if results['avg_word_length'] > 4.8:
        score += 1
    if results['lexical_diversity'] >= 0.5:
        score += 1
    print(score)
    #классификация
    if score <= 2:
        return "A1-A2 (Beginner)"
    elif score <= 4:
        return "B1 (Intermediate)"
    elif score <= 6:
        return "B2 (Upper-Intermediate)"
    else:
        return "C1-C2 (Advanced)"

In [132]:
def read_text_from_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            text = file.read()
        return text
    except FileNotFoundError:
        print(f"Ошибка: Файл '{file_path}' не найден.")
        return ""
    except Exception as e:
        print(f"Произошла ошибка при чтении файла: {e}")
        return ""

In [134]:
file_path = "C:/Users/boss/Documents/examples/test.txt"  # Укажите путь к вашему файлу
text_from_file = read_text_from_file(file_path)

if text_from_file:  # Если файл успешно прочитан
    analysis = analyze_text(text_from_file)
    level = estimate_level(analysis)

    print("Результаты анализа текста из файла:")
    for key, value in analysis.items():
        print(f"{key}: {value}")
    print(f"\nОценочный уровень английского: {level}")

9
Результаты анализа текста из файла:
total_words: 612
avg_word_length: 4.46
A1_level_words: 0.51
A2_level_words: 0.1
B1_level_words: 0.05
B2_level_words: 0.05
C1_level_words: 0.01
C2_level_words: 0.0
vocabulary_size: 326
lexical_diversity: 0.53
complex_sentence_ratio: 0.5454545454545454
passive_voice_ratio: 0.0032679738562091504
flesch_reading_ease: 60.91207977207978
smog_index: 11.715833242495528
coleman_liau_index: 8.778396072013095
sentence_count: 27
lexicon_count: 611

Оценочный уровень английского: C1-C2 (Advanced)
